In [13]:
#Imports
import os
import re
import numpy as np
import pandas as pd
import requests
import tqdm

In [14]:
#Download Raw data from Github repo
print("Downloading data from GitHub...")

# GitHub API URL for the directory
api_url = "https://api.github.com/repos/COMP3608-GROUP-9-PROJECT/COMP3608_PROJECT/contents/RawData"

# Create local directory
os.makedirs("raw", exist_ok=True)

# Get list of files from GitHub API
response = requests.get(api_url)
if response.status_code == 200:
    files = response.json()
    for file in files:
        if file['type'] == 'file' and file['name'].endswith('.csv'): # Check if it's a file and ends with .csv
            download_url = file['download_url']
            r = requests.get(download_url)
            with open(f"raw/{file['name']}", "wb") as f:
                f.write(r.content)
            print(f"Downloaded: {file['name']}")
else:
    print(f"Failed to access API: {response.status_code}")

print("Success: All CSV files downloaded.\n")

Downloaded: Cities.csv
Downloaded: Conferences.csv
Downloaded: MConferenceTourneyGames.csv
Downloaded: MGameCities.csv
Downloaded: MMasseyOrdinals.csv
Downloaded: MNCAATourneyCompactResults.csv
Downloaded: MNCAATourneyDetailedResults.csv
Downloaded: MNCAATourneySeedRoundSlots.csv
Downloaded: MNCAATourneySeeds.csv
Downloaded: MNCAATourneySlots.csv
Downloaded: MRegularSeasonCompactResults.csv
Downloaded: MRegularSeasonDetailedResults.csv
Downloaded: MSeasons.csv
Downloaded: MSecondaryTourneyCompactResults.csv
Downloaded: MSecondaryTourneyTeams.csv
Downloaded: MTeamCoaches.csv
Downloaded: MTeamConferences.csv
Downloaded: MTeamSpellings.csv
Downloaded: MTeams.csv
Downloaded: SampleSubmissionStage1.csv
Downloaded: SampleSubmissionStage2.csv
Downloaded: SeedBenchmarkStage1.csv
Downloaded: WConferenceTourneyGames.csv
Downloaded: WGameCities.csv
Downloaded: WNCAATourneyCompactResults.csv
Downloaded: WNCAATourneyDetailedResults.csv
Downloaded: WNCAATourneySeeds.csv
Downloaded: WNCAATourneySlots

In [15]:
#Process CSV files
print("Processing local CSV files...")

# Mapping the internal variable names to their respective file pairs
data_map = {
    "seasonResults": ("MRegularSeasonDetailedResults.csv", "WRegularSeasonDetailedResults.csv"),
    "tourneyResults": ("MNCAATourneyDetailedResults.csv", "WNCAATourneyDetailedResults.csv"),
    "seeds": ("MNCAATourneySeeds.csv", "WNCAATourneySeeds.csv")
}

# Dictionary to temporarily hold our loaded dataframes
dfs = {}

for var_name, (men_file, women_file) in data_map.items():
    # Load and tag Men's data (Divison 1)
    m_df = pd.read_csv(f"raw/{men_file}").assign(Division=1)
    # Load and tag Women's data (Divison 0)
    w_df = pd.read_csv(f"raw/{women_file}").assign(Division=0)

    # Combine and store
    dfs[var_name] = pd.concat([m_df, w_df], ignore_index=True)

# Unpack the dictionary into specific variables
seasonResults = dfs["seasonResults"]
tourneyResults = dfs["tourneyResults"]
seeds = dfs["seeds"]

# Seeds adjustments

# tourneyResults Seasons start in 2003
seeds = seeds[seeds["Season"] >= 2003].copy()

# Convert Seed (e.g., 'W01') to integer (1)
seeds["Seed"] = seeds["Seed"].str[1:3].astype(int)


print(f"Season Results merged: {seasonResults.shape}")
print(f"Tourney Results merged: {tourneyResults.shape}")
print(f"Seed data prepared:    {seeds.shape}\n")

Processing local CSV files...
Season Results merged: (200590, 35)
Tourney Results merged: (2276, 35)
Seed data prepared:    (2896, 4)



In [16]:
seasonResults

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF,Division
0,2003,10,1104,68,1328,62,N,0,27,58,...,16,22,10,22,8,18,9,2,20,1
1,2003,10,1272,70,1393,63,N,0,26,62,...,9,20,20,25,7,12,8,6,16,1
2,2003,11,1266,73,1437,61,N,0,24,58,...,14,23,31,22,9,12,2,5,23,1
3,2003,11,1296,56,1457,50,N,0,18,38,...,8,15,17,20,9,19,4,3,23,1
4,2003,11,1400,77,1208,71,N,0,30,61,...,17,27,21,15,12,10,7,1,14,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
200585,2025,131,3471,75,3413,66,A,0,26,62,...,9,14,9,26,14,10,6,5,22,0
200586,2025,132,3192,66,3476,49,H,0,23,55,...,3,4,14,22,14,17,4,1,17,0
200587,2025,132,3250,74,3119,62,H,0,27,45,...,6,10,8,13,10,10,5,0,20,0
200588,2025,132,3293,83,3125,62,N,0,28,54,...,12,14,12,22,11,7,5,0,16,0


In [17]:
tourneyResults

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF,Division
0,2003,134,1421,92,1411,84,N,1,32,69,...,14,31,17,28,16,15,5,0,22,1
1,2003,136,1112,80,1436,51,N,0,31,66,...,7,7,8,26,12,17,10,3,15,1
2,2003,136,1113,84,1272,71,N,0,31,59,...,14,21,20,22,11,12,2,5,18,1
3,2003,136,1141,79,1166,73,N,0,29,53,...,12,17,14,17,20,21,6,6,21,1
4,2003,136,1143,76,1301,74,N,1,27,64,...,15,20,10,26,16,14,5,8,19,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2271,2024,147,3163,80,3425,73,A,0,28,58,...,18,20,10,25,10,9,6,4,20,0
2272,2024,147,3234,94,3261,87,H,0,32,69,...,11,17,21,28,15,13,6,6,21,0
2273,2024,151,3234,71,3163,69,N,0,27,59,...,3,4,6,22,21,14,15,1,18,0
2274,2024,151,3376,78,3301,59,N,0,33,66,...,13,18,10,18,5,12,9,1,9,0


In [18]:
seeds

,Season,Seed,TeamID,Division
1154,2003,1,1328,1
1155,2003,2,1448,1
1156,2003,3,1393,1
1157,2003,4,1257,1
1158,2003,5,1280,1
...,...,...,...,...
4365,2025,12,3193,0
4366,2025,13,3251,0
4367,2025,14,3195,0
4368,2025,15,3117,0


In [19]:
# Data preparation

print("Normalizing pace and structuring symmetric game pairings...")

def format_game_data(df):
    # Group the columns logically for better readability
    meta_cols = ["Season", "DayNum", "NumOT", "Division"]
    win_cols = ["WTeamID", "WScore", "WFGM", "WFGA", "WFGM3", "WFGA3", "WFTM", "WFTA",
                "WOR", "WDR", "WAst", "WTO", "WStl", "WBlk", "WPF"]
    loss_cols = ["LTeamID", "LScore", "LFGM", "LFGA", "LFGM3", "LFGA3", "LFTM", "LFTA",
                 "LOR", "LDR", "LAst", "LTO", "LStl", "LBlk", "LPF"]

    # Isolate the required dataframe
    df = df[meta_cols + win_cols + loss_cols].copy()

    # Adjustments

    # Normalize stats to a standard 40-minute regulation game
    # Overtime is 5 mins. Pace Factor = (40 + 5 * OT) / 40
    pace_factor = 1 + (df["NumOT"] * 5 / 40)

    # Dynamically grab all columns that are game stats (excluding metadata and Team IDs)
    stat_cols = [c for c in df.columns if c not in meta_cols + ["WTeamID", "LTeamID"]]

    # Apply pace adjustment
    for col in stat_cols:
        df[col] = df[col] / pace_factor

    # Create symmetric Team 1 (T1) vs Team 2 (T2) pairings
    # Perspective 1: T1 is the Winner, T2 is the Loser
    df_p1 = df.copy()
    df_p1.columns = [c.replace("W", "T1").replace("L", "T2") for c in df.columns]

    # Perspective 2: T1 is the Loser, T2 is the Winner
    df_p2 = df.copy()
    df_p2.columns = [c.replace("L", "T1").replace("W", "T2") for c in df.columns]

    # Stack them on top of each other
    symmetric_df = pd.concat([df_p1, df_p2], ignore_index=True)

    # 3. Target Variable and Context Features
    symmetric_df["PointDifference"] = symmetric_df["T1Score"] - symmetric_df["T2Score"]
    symmetric_df["Win"] = (symmetric_df["PointDifference"] > 0).astype(int)

    # Vectorized string check to identify Division (Men's IDs start with '1', Women's with '3')
    symmetric_df["Division"] = symmetric_df["T1TeamID"].astype(str).str.startswith("1").astype(int)

    return symmetric_df

# Apply the function to our datasets
seasonData = format_game_data(seasonResults)
tourneyData = format_game_data(tourneyResults)

print(f"Symmetric Season Data:  {seasonData.shape}")
print(f"Symmetric Tourney Data: {tourneyData.shape}\n")

Normalizing pace and structuring symmetric game pairings...
Symmetric Season Data:  (401180, 36)
Symmetric Tourney Data: (4552, 36)



In [20]:
seasonData

,Season,DayNum,NumOT,Division,T1TeamID,T1Score,T1FGM,T1FGA,T1FGM3,T1FGA3,...,T2FTA,T2OR,T2DR,T2Ast,T2TO,T2Stl,T2Blk,T2PF,PointDifference,Win
0,2003,10,0,1,1104,68.0,27.0,58.0,3.0,14.0,...,22.0,10.0,22.0,8.0,18.0,9.0,2.0,20.0,6.0,1
1,2003,10,0,1,1272,70.0,26.0,62.0,8.0,20.0,...,20.0,20.0,25.0,7.0,12.0,8.0,6.0,16.0,7.0,1
2,2003,11,0,1,1266,73.0,24.0,58.0,8.0,18.0,...,23.0,31.0,22.0,9.0,12.0,2.0,5.0,23.0,12.0,1
3,2003,11,0,1,1296,56.0,18.0,38.0,3.0,9.0,...,15.0,17.0,20.0,9.0,19.0,4.0,3.0,23.0,6.0,1
4,2003,11,0,1,1400,77.0,30.0,61.0,6.0,14.0,...,27.0,21.0,15.0,12.0,10.0,7.0,1.0,14.0,6.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401175,2025,131,0,0,3413,66.0,24.0,67.0,9.0,29.0,...,28.0,8.0,31.0,10.0,11.0,6.0,1.0,20.0,-9.0,0
401176,2025,132,0,0,3476,49.0,21.0,57.0,4.0,20.0,...,18.0,10.0,22.0,11.0,9.0,8.0,1.0,8.0,-17.0,0
401177,2025,132,0,0,3119,62.0,25.0,56.0,6.0,17.0,...,17.0,5.0,25.0,15.0,15.0,6.0,0.0,12.0,-12.0,0
401178,2025,132,0,0,3125,62.0,24.0,68.0,2.0,21.0,...,15.0,5.0,33.0,21.0,13.0,2.0,3.0,15.0,-21.0,0


In [21]:
tourneyData

,Season,DayNum,NumOT,Division,T1TeamID,T1Score,T1FGM,T1FGA,T1FGM3,T1FGA3,...,T2FTA,T2OR,T2DR,T2Ast,T2TO,T2Stl,T2Blk,T2PF,PointDifference,Win
0,2003,134,1,1,1421,81.777778,28.444444,61.333333,9.777778,25.777778,...,27.555556,15.111111,24.888889,14.222222,13.333333,4.444444,0.000000,19.555556,7.111111,1
1,2003,136,0,1,1112,80.000000,31.000000,66.000000,7.000000,23.000000,...,7.000000,8.000000,26.000000,12.000000,17.000000,10.000000,3.000000,15.000000,29.000000,1
2,2003,136,0,1,1113,84.000000,31.000000,59.000000,6.000000,14.000000,...,21.000000,20.000000,22.000000,11.000000,12.000000,2.000000,5.000000,18.000000,13.000000,1
3,2003,136,0,1,1141,79.000000,29.000000,53.000000,3.000000,7.000000,...,17.000000,14.000000,17.000000,20.000000,21.000000,6.000000,6.000000,21.000000,6.000000,1
4,2003,136,1,1,1143,67.555556,24.000000,56.888889,6.222222,17.777778,...,17.777778,8.888889,23.111111,14.222222,12.444444,4.444444,7.111111,16.888889,1.777778,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4547,2024,147,0,0,3425,73.000000,23.000000,70.000000,9.000000,29.000000,...,27.000000,5.000000,30.000000,17.000000,12.000000,6.000000,5.000000,21.000000,-7.000000,0
4548,2024,147,0,0,3261,87.000000,34.000000,88.000000,8.000000,24.000000,...,22.000000,3.000000,29.000000,16.000000,11.000000,6.000000,3.000000,15.000000,-7.000000,0
4549,2024,151,0,0,3163,69.000000,29.000000,63.000000,8.000000,25.000000,...,14.000000,9.000000,23.000000,12.000000,16.000000,7.000000,1.000000,9.000000,-2.000000,0
4550,2024,151,0,0,3301,59.000000,20.000000,62.000000,6.000000,23.000000,...,4.000000,10.000000,34.000000,18.000000,15.000000,10.000000,6.000000,16.000000,-19.000000,0


In [22]:
# Aggregate season performance

print("Aggregating season teams score performance...")

# Numeric columns we want to average out
# Include both the teams stats and what they allowed their opponents to do
stat_categories = [
    "T1Score", "T1FGM", "T1FGA", "T1FGM3", "T1FGA3", "T1FTM", "T1FTA",
    "T1OR", "T1DR", "T1Ast", "T1TO", "T1Stl", "T1Blk", "T1PF",
    "T2Score", "T2FGM", "T2FGA", "T2FGM3", "T2FGA3", "T2FTM", "T2FTA",
    "T2OR", "T2DR", "T2Ast", "T2TO", "T2Stl", "T2Blk", "T2PF",
    "PointDifference"
]

# Calculate the mean for every team, every season
seasonal_summary = seasonData.groupby(["Season", "T1TeamID"])[stat_categories].mean().reset_index()

print(f"Calculated averages for {seasonal_summary.shape[0]} team-seasons")

# Function to rename columns with proper perspective labels
# This turns 'T1Score' into 'T1AverageScore' and 'T2Score' into 'T1AverageOpponentScore'
def rename_perspective(df, team_col, prefix):
    new_cols = {}
    for col in df.columns:
        if col in ["Season", team_col]:
            # Keep these as is
            continue
        elif col.startswith("T1"):
            # Team's own stats: T1Score ==> T1AverageScore
            clean_name = col.replace("T1", "")
            new_cols[col] = f"{prefix}{clean_name}"
        elif col.startswith("T2"):
            # Opponent stats: T2Score ==> T1AverageOpponentScore
            clean_name = col.replace("T2", "")
            new_cols[col] = f"{prefix}Opponent{clean_name}"
        elif col == "PointDifference":
            new_cols[col] = f"{prefix}PointDifference"

    return df.rename(columns=new_cols)

# Create Team 1 perspective (for merging on T1TeamID in tournament data)
season_avgs_t1 = seasonal_summary.copy()
season_avgs_t1 = rename_perspective(season_avgs_t1, "T1TeamID", "T1Average")

# Create Team 2 perspective (for merging on T2TeamID in tournament data)
season_avgs_t2 = seasonal_summary.copy()
season_avgs_t2 = season_avgs_t2.rename(columns={"T1TeamID": "T2TeamID"})
season_avgs_t2 = rename_perspective(season_avgs_t2, "T2TeamID", "T2Average")

print(f"Created {season_avgs_t1.shape[0]} unique team-season profiles")
print(f"Team 1 perspective: {len(season_avgs_t1.columns)} columns")
print(f"Team 2 perspective: {len(season_avgs_t2.columns)} columns")

Aggregating season teams score performance...
Calculated averages for 13583 team-seasons
Created 13583 unique team-season profiles
Team 1 perspective: 31 columns
Team 2 perspective: 31 columns


In [23]:
seasonal_summary

,Season,T1TeamID,T1Score,T1FGM,T1FGA,T1FGM3,T1FGA3,T1FTM,T1FTA,T1OR,...,T2FTM,T2FTA,T2OR,T2DR,T2Ast,T2TO,T2Stl,T2Blk,T2PF,PointDifference
0,2003,1102,57.250000,19.142857,39.785714,7.821429,20.821429,11.142857,17.107143,4.178571,...,13.678571,19.250000,9.607143,20.142857,9.142857,12.964286,5.428571,1.571429,18.357143,0.250000
1,2003,1103,75.967901,26.268313,54.051029,5.296296,15.611523,18.134979,24.580247,9.497119,...,15.245267,21.069136,11.443621,21.232922,14.907819,14.753909,6.218930,2.750617,21.566255,0.708642
2,2003,1104,69.015873,23.940476,56.964286,6.329365,19.805556,14.805556,20.849206,13.523810,...,12.107143,17.095238,10.857143,22.551587,11.638889,13.789683,5.503968,3.170635,19.170635,4.261905
3,2003,1105,70.400855,23.938462,60.519658,7.447009,20.373504,15.076923,21.348718,13.252137,...,16.023077,23.894872,12.901709,25.925641,15.561538,18.485470,9.217949,4.138462,18.683761,-4.843590
4,2003,1106,63.281746,23.309524,55.019841,6.083333,17.567460,10.579365,16.384921,12.234127,...,15.468254,21.876984,11.273810,22.273810,11.722222,15.003968,8.773810,3.154762,16.079365,-0.162698
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13578,2025,3476,63.341935,23.722581,57.000000,7.141935,21.522581,8.754839,10.870968,9.019355,...,10.419355,14.277419,9.025806,21.161290,12.658065,12.748387,9.006452,2.941935,15.503226,-2.658065
13579,2025,3477,60.275720,22.267490,59.090535,7.024691,21.765432,8.716049,12.111111,6.864198,...,12.378601,18.061728,11.205761,27.255144,14.345679,12.934156,8.131687,3.004115,12.823045,-11.283951
13580,2025,3478,53.379928,17.974910,50.996416,6.254480,20.028674,11.175627,14.856631,7.243728,...,7.931900,10.792115,9.770609,23.498208,16.498208,11.767025,7.738351,2.537634,15.530466,-15.096774
13581,2025,3479,61.763889,21.143519,51.930556,6.560185,20.555556,12.916667,17.921296,5.185185,...,12.814815,18.129630,8.273148,23.995370,12.203704,15.125000,8.773148,2.328704,19.347222,-7.712963


In [24]:
# Merge tournament seeds and seasonal score performance

print("Merging tournament seeds and seasonal performance metrics...")

# Prepare seed lookup tables for both sides of the matchup
# Rename columns to alignn with our T1/T2 symmetric structure
t1_seeds = seeds[['Season', 'TeamID', 'Seed']].rename(columns={'TeamID': 'T1TeamID', 'Seed': 'T1Seed'})
t2_seeds = seeds[['Season', 'TeamID', 'Seed']].rename(columns={'TeamID': 'T2TeamID', 'Seed': 'T2Seed'})

# Isolate core tournament results for merging
# Note: Keept 'Divison' as per previous steps for consistency
tourney_base = tourneyData[['Season', 'T1TeamID', 'T2TeamID', 'PointDifference', 'Win', 'Division']].copy()

# Attach Seed Information
tourney_base = tourney_base.merge(t1_seeds, on=['Season', 'T1TeamID'], how='left')
tourney_base = tourney_base.merge(t2_seeds, on=['Season', 'T2TeamID'], how='left')

# Calculate the relative strength gap (Seed Difference)
# Positive value means Team 1 is the higher (better) seed
tourney_base["SeedDifference"] = tourney_base["T2Seed"] - tourney_base["T1Seed"]

# Attach Seasonal Averages
tourney_base = tourney_base.merge(season_avgs_t1, on=['Season', 'T1TeamID'], how='left')
tourney_base = tourney_base.merge(season_avgs_t2, on=['Season', 'T2TeamID'], how='left')

tourneyData = tourney_base

print(f"Seeds and Season Averages successfully integrated.")
print(f"Final feature count for tournament matchups: {len(tourneyData.columns)}")
print(f"Total tournament rows for training: {tourneyData.shape[0]}\n")

Merging tournament seeds and seasonal performance metrics...
Seeds and Season Averages successfully integrated.
Final feature count for tournament matchups: 67
Total tournament rows for training: 4552



In [25]:
tourneyData

,Season,T1TeamID,T2TeamID,PointDifference,Win,Division,T1Seed,T2Seed,SeedDifference,T1AverageScore,...,T2AverageOpponentFTM,T2AverageOpponentFTA,T2AverageOpponentOR,T2AverageOpponentDR,T2AverageOpponentAst,T2AverageOpponentTO,T2AverageOpponentStl,T2AverageOpponentBlk,T2AverageOpponentPF,T2AveragePointDifference
0,2003,1421,1411,7.111111,1,1,16,16,0,69.615326,...,11.914815,18.655556,11.881481,22.781481,13.718519,14.259259,7.977778,2.596296,21.533333,1.948148
1,2003,1112,1436,29.000000,1,1,1,16,15,84.511905,...,10.331034,15.482759,9.517241,21.641379,13.158621,12.910345,7.082759,3.655172,17.772414,4.689655
2,2003,1113,1272,13.000000,1,1,10,7,-3,75.344828,...,13.333333,20.659004,12.295019,23.482759,13.237548,15.019157,7.252874,3.153257,19.827586,8.693487
3,2003,1141,1166,6.000000,1,1,11,6,-5,79.344828,...,11.643098,16.619529,11.020202,21.289562,12.329966,17.006734,6.306397,2.569024,19.323232,14.898990
4,2003,1143,1301,1.777778,1,1,8,9,1,73.636015,...,15.374074,21.129630,10.514815,21.348148,12.511111,14.581481,7.418519,2.811111,19.262963,4.370370
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4547,2024,3425,3163,-7.000000,0,0,1,3,2,74.064516,...,8.666667,12.242424,7.484848,19.636364,11.757576,15.969697,5.212121,2.181818,16.303030,22.909091
4548,2024,3261,3234,-7.000000,0,0,3,1,-2,86.696970,...,11.050505,14.841751,8.740741,21.531987,14.087542,14.333333,7.010101,2.811448,19.114478,20.949495
4549,2024,3163,3234,-2.000000,0,0,3,1,-2,79.878788,...,11.050505,14.841751,8.740741,21.531987,14.087542,14.333333,7.010101,2.811448,19.114478,20.949495
4550,2024,3301,3376,-19.000000,0,0,3,1,-2,72.979798,...,9.593750,14.031250,9.250000,19.281250,10.062500,14.812500,6.625000,3.125000,16.812500,29.750000


In [26]:
# Elo-based strength ranked


print("Simulating season ELO progression...")

# Elo Constants
START_ELO = 1000
K_FACTOR = 100  # How much a single game changes a rating
SCALE_FACTOR = 400

def get_expected_outcome(rating_a, rating_b):
    """Calculates the probability that Team A beats Team B."""
    return 1.0 / (1 + 10 ** ((rating_b - rating_a) / SCALE_FACTOR))

def update_elo(winner_val, loser_val):
    """Calculates the rating shift based on game outcome."""
    expected = get_expected_outcome(winner_val, loser_val)
    shift = K_FACTOR * (1 - expected)
    return winner_val + shift, loser_val - shift

seasonal_elo_list = []

# Process each season independently
for season_year in sorted(seasonData["Season"].unique()):
    # Filter for regular season games where Team 1 won (to avoid double-counting)
    matchups = seasonData.query("Season == @season_year and Win == 1")[["T1TeamID", "T2TeamID"]]

    # Initialize all teams in the season at the baseline rating
    unique_teams = set(matchups["T1TeamID"]).union(set(matchups["T2TeamID"]))
    current_ratings = {team: START_ELO for team in unique_teams}

    # Sequential update, Iterating through games (using itertuples for speed)
    for row in matchups.itertuples(index=False):
        w_id, l_id = row.T1TeamID, row.T2TeamID

        # Calculate new ratings
        new_w, new_l = update_elo(current_ratings[w_id], current_ratings[l_id])

        # Update the season tracker
        current_ratings[w_id] = new_w
        current_ratings[l_id] = new_l

    # Convert dictionary to DataFrame for this season
    season_elo_df = pd.DataFrame.from_dict(current_ratings, orient="index").reset_index()
    season_elo_df.columns = ["TeamID", "Rating"]
    season_elo_df["Season"] = season_year
    seasonal_elo_list.append(season_elo_df)

# Combine all seasons into one master Elo lookup table
full_elo_df = pd.concat(seasonal_elo_list, ignore_index=True)

# 1. Merge Elo for Team 1
tourneyData = tourneyData.merge(
    full_elo_df.rename(columns={"TeamID": "T1TeamID", "Rating": "T1Rating"}),
    on=["Season", "T1TeamID"], how="left"
)

# 2. Merge Elo for Team 2
tourneyData = tourneyData.merge(
    full_elo_df.rename(columns={"TeamID": "T2TeamID", "Rating": "T2Rating"}),
    on=["Season", "T2TeamID"], how="left"
)

# 3. Create the 'Rating Gap' feature
tourneyData["RatingDifference"] = tourneyData["T1Rating"] - tourneyData["T2Rating"]

print(f"ELO simulation complete.")
print(f"Feature 'RatingDifference' added to training set.\n")

Simulating season ELO progression...
ELO simulation complete.
Feature 'RatingDifference' added to training set.



In [27]:
sample_submission = pd.read_csv(os.path.join("raw", "SampleSubmissionStage2.csv"))

# Extract season and team IDs from the sample submission file
sample_submission[['Season', 'T1TeamID', 'T2TeamID']] = sample_submission['ID'].str.split('_', expand=True).astype(int)

# Determine division based on team IDs (IDs >= 2000 are women, IDs < 2000 are men)
sample_submission['Division'] = (sample_submission['T1TeamID'] >= 2000).astype(int)



In [28]:
# Merge season averages with the sample submission

# Key columns for merging

t1_keys = ['Season', 'T1TeamID']
t2_keys = ['Season', 'T2TeamID']

# Remove columns that already exist in sample submisssion to avoid duplication

season_avgs_t1_clean = season_avgs_t1[
    [col for col in season_avgs_t1.columns if col in t1_keys or col not in sample_submission.columns]
]

season_avgs_t2_clean = season_avgs_t2[
    [col for col in season_avgs_t2.columns if col in t2_keys or col not in sample_submission.columns]
]

# Merge season averages into sample submission

sample_submission = sample_submission.merge(season_avgs_t1_clean, on=t1_keys, how='left')
sample_submission = sample_submission.merge(season_avgs_t2_clean, on=t2_keys, how='left')

In [29]:
stat_bases = [
    "Score", "FGM", "FGA", "FGM3", "FGA3", "FTM", "FTA",
    "OR", "DR", "Ast", "TO", "Stl", "Blk", "PF"
]

for stat in stat_bases:
    tourneyData[f"{stat}_diff"] = tourneyData[f"T1Average{stat}"] - tourneyData[f"T2Average{stat}"]
    sample_submission[f"{stat}_diff"] = sample_submission[f"T1Average{stat}"] - sample_submission[f"T2Average{stat}"]


# Drop all columns starting with 'T1' or 'T2' as they have been replaced by difference features

cols_to_drop = [col for col in tourneyData.columns if col.startswith("T1") or col.startswith("T2")]
cols_to_drop = [col for col in cols_to_drop if col not in ["Win", "PointDifference"]]
tourneyData = tourneyData.drop(columns=cols_to_drop)

drop_cols = [col for col in sample_submission.columns if col.startswith("T1") or col.startswith("T2")]
sample_submission = sample_submission.drop(columns=drop_cols)

print(f"Difference features created and original T1/T2 columns dropped.")
print(f"Final feature count for processed tournament matchups: {len(tourneyData.columns)}\n")

Difference features created and original T1/T2 columns dropped.
Final feature count for processed tournament matchups: 20



In [30]:
sample_submission

,ID,Pred,Season,Division,Score_diff,FGM_diff,FGA_diff,FGM3_diff,FGA3_diff,FTM_diff,FTA_diff,OR_diff,DR_diff,Ast_diff,TO_diff,Stl_diff,Blk_diff,PF_diff
0,2025_1101_1102,0.5,2025,0,5.522749,2.879789,5.595666,-3.964080,-10.172294,3.727251,3.109076,2.685105,-1.363266,-0.899545,1.961686,4.331777,0.047534,3.562979
1,2025_1101_1103,0.5,2025,0,-16.761973,-6.144516,-8.126557,-6.571719,-14.887572,2.098779,3.650742,-1.346145,-5.349377,-5.253712,2.371408,2.415110,-0.830939,3.021312
2,2025_1101_1104,0.5,2025,0,-23.575525,-6.921978,-8.272495,-6.313828,-15.524672,-3.417741,-4.836526,-2.318472,-8.753628,-4.312319,2.042494,3.916057,-1.546848,2.385580
3,2025_1101_1105,0.5,2025,0,-1.190526,1.209335,-3.112504,-3.454197,-10.354232,-0.154998,-1.852316,-2.562522,-0.235806,0.215604,-0.677813,2.082550,-0.827238,0.258447
4,2025_1101_1106,0.5,2025,0,-4.888657,-1.022989,-6.821317,-4.741437,-12.699756,1.898758,2.109602,-0.739347,-3.016254,1.037850,5.358992,2.336933,0.005341,3.126321
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131402,2025_3477_3479,0.5,2025,1,-1.488169,1.123971,7.159979,0.464506,1.209877,-4.200617,-5.810185,1.679012,0.846193,2.698560,-2.436728,0.582305,1.966049,-1.309156
131403,2025_3477_3480,0.5,2025,1,-4.287772,-0.887272,1.340535,1.258818,1.174162,-3.772046,-6.492063,-3.373898,0.132569,0.551734,-1.436067,-1.016902,0.850970,0.016902
131404,2025_3478_3479,0.5,2025,1,-8.383961,-3.168608,-0.934140,-0.305705,-0.526882,-1.741039,-3.064665,2.058542,-0.622013,-0.663978,-1.854092,-0.388889,0.533154,-4.857676
131405,2025_3478_3480,0.5,2025,1,-11.183564,-5.179852,-6.753584,0.488607,-0.562596,-1.312468,-3.746544,-2.994368,-1.335637,-2.810804,-0.853431,-1.988095,-0.581925,-3.531618


In [31]:
tourneyData

,Season,PointDifference,Win,Division,SeedDifference,RatingDifference,Score_diff,FGM_diff,FGA_diff,FGM3_diff,FGA3_diff,FTM_diff,FTA_diff,OR_diff,DR_diff,Ast_diff,TO_diff,Stl_diff,Blk_diff,PF_diff
0,2003,7.111111,1,1,0,-1.557555,-2.918008,-0.796935,0.587995,0.437548,-0.778799,-1.761686,-7.501277,-1.092209,-2.024904,-1.400511,0.764368,0.520562,0.714943,0.558238
1,2003,29.000000,1,1,15,264.691130,17.256732,5.432978,9.659715,1.742748,4.546798,4.648030,5.434182,2.148139,1.874959,3.391544,0.673563,1.555446,1.227887,1.883853
2,2003,13.000000,1,1,-3,-175.726122,1.134100,0.819923,-3.325670,-2.992337,-7.521073,2.486590,3.111111,-0.517241,-2.735632,-1.107280,0.141762,-2.199234,-0.850575,0.567050
3,2003,6.000000,1,1,-5,-24.479498,0.338094,-1.998839,-4.599907,-1.115175,-2.493208,5.450946,5.219552,-0.262278,0.178219,-1.163822,4.931615,-1.266922,-0.427609,3.760130
4,2003,1.777778,1,1,1,-62.930889,1.513793,2.767178,4.816731,-1.626564,-5.640358,-2.393997,-1.013538,1.414943,2.188378,1.201277,-0.110473,-1.281098,-0.289527,-1.714815
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4547,2024,-7.000000,0,0,2,14.505158,-5.814272,-4.242229,0.157185,0.412317,1.302444,2.257869,3.166764,2.233627,-1.550147,-4.593353,-0.647507,-1.934311,1.831476,1.629912
4548,2024,-7.000000,0,0,-2,-13.701969,-5.525253,-1.696970,1.468013,-6.983165,-16.188552,4.851852,7.306397,5.912458,-0.410774,-5.175084,1.888889,3.279461,2.185185,1.525253
4549,2024,-2.000000,0,0,-2,23.583548,-12.343434,-2.363636,-3.713805,-4.134680,-9.612795,-3.481481,-3.936027,-0.936027,-2.804714,-2.417508,-0.959596,2.309764,0.488215,-0.202020
4550,2024,-19.000000,0,0,-2,-301.178457,-13.082702,-6.538615,-4.620581,-0.472117,2.145412,0.466646,-0.712753,-3.143624,-0.560501,-5.420455,-0.934028,-2.325758,-3.636364,-1.861322


In [32]:
# Save dataset
print("Exporting dataset ...")

output_dir = "ProcessedData"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "processed_tournament_data.csv")
output_pathSample = os.path.join(output_dir, "Sample_Submission.csv")


# Export the dataframe to CSV
# index=False so Pandas doesn't write the row numbers as an extra column
tourneyData.to_csv(output_path, index=False)
sample_submission.to_csv(output_pathSample, index=False)

print(f"Success: Data saved to: {output_path}")
print(f"Final Dataset Dimensions: {tourneyData.shape[0]} rows | {tourneyData.shape[1]} features")

Exporting dataset ...
Success: Data saved to: ProcessedData\processed_tournament_data.csv
Final Dataset Dimensions: 4552 rows | 20 features
